In [ ]:
%pip install nltk pandas matplotlib scikit-learn torch rouge-score -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
import re
import os
import json
from pathlib import Path

from nltk.tokenize import wordpunct_tokenize
from collections import Counter
from sklearn.metrics import precision_recall_fscore_support, classification_report
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer

def tokenize(text):
    return wordpunct_tokenize(str(text).lower())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

: 

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dataset").exists() and (PROJECT_ROOT.parent / "dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "dataset"
RESULTS_DIR = PROJECT_ROOT / "results"

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in (train_df, val_df, test_df):
    df["clean_text"] = df["clean_text"].fillna("").astype(str)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
train_df["clean_text"].head(3)

In [ ]:
VOCAB_SIZE  = 5000
MAX_SEQ_LEN = 30       # most tweets fit within 30 tokens
PAD_TOKEN   = "<PAD>"
UNK_TOKEN   = "<UNK>"
SOS_TOKEN   = "<SOS>"
EOS_TOKEN   = "<EOS>"

all_tokens = []
for text in train_df["clean_text"]:
    all_tokens.extend(tokenize(text))

counter = Counter(all_tokens)
most_common = [tok for tok, _ in counter.most_common(VOCAB_SIZE - 4)]

vocab = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN] + most_common
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

PAD_IDX = word2idx[PAD_TOKEN]
UNK_IDX = word2idx[UNK_TOKEN]
SOS_IDX = word2idx[SOS_TOKEN]
EOS_IDX = word2idx[EOS_TOKEN]

print(f"Vocabulary size: {len(vocab)}")

In [ ]:
def encode(text, max_len=MAX_SEQ_LEN):
    tokens = tokenize(text)[:max_len]
    ids = [SOS_IDX] + [word2idx.get(t, UNK_IDX) for t in tokens] + [EOS_IDX]
    # Pad or truncate to fixed length
    ids = ids[:max_len]
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids

def decode(ids):
    tokens = []
    for i in ids:
        tok = idx2word.get(i, UNK_TOKEN)
        if tok in (PAD_TOKEN, SOS_TOKEN, EOS_TOKEN):
            continue
        tokens.append(tok)
    return " ".join(tokens)

# Encode all splits
train_encoded = [encode(t) for t in train_df["clean_text"]]
test_encoded  = [encode(t) for t in test_df["clean_text"]]

print(f"Sample encoded: {train_encoded[0]}")
print(f"Sample decoded: {decode(train_encoded[0])}")

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, encoded_sequences):
        self.data = torch.tensor(encoded_sequences, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

BATCH_SIZE = 64

train_dataset = TweetDataset(train_encoded)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True, drop_last=True)

print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
class Generator(nn.Module):
    """
    Noise vector → GRU → linear projection over vocab
    Outputs a sequence of token logits (one per timestep).
    """
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128,
                 seq_len=MAX_SEQ_LEN, noise_dim=100):
        super().__init__()
        self.seq_len    = seq_len
        self.hidden_dim = hidden_dim
        self.noise_dim  = noise_dim

        self.fc_noise = nn.Linear(noise_dim, hidden_dim)
        self.embed    = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.gru      = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.out      = nn.Linear(hidden_dim, vocab_size)

    def forward(self, noise, temperature=1.0):
        batch = noise.size(0)

        # Project noise into initial hidden state
        h0 = torch.tanh(self.fc_noise(noise)).unsqueeze(0)  # (1, B, H)

        # Start token for every item in batch
        inp = torch.full((batch, 1), SOS_IDX,
                         dtype=torch.long, device=noise.device)

        logits_seq = []
        token_seq  = []

        for _ in range(self.seq_len):
            emb, h0     = self.gru(self.embed(inp), h0)    # (B,1,H)
            logits      = self.out(emb.squeeze(1))          # (B, V)
            logits_seq.append(logits.unsqueeze(1))

            # Gumbel-softmax straight-through for differentiable token selection
            probs = torch.softmax(logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, 1)          # (B, 1)
            token_seq.append(next_tok)
            inp = next_tok

        logits_all = torch.cat(logits_seq, dim=1)   # (B, T, V)
        tokens_all = torch.cat(token_seq,  dim=1)   # (B, T)
        return logits_all, tokens_all

In [ ]:
class Discriminator(nn.Module):
    """
    Token sequence → Embedding → 1D-CNN → sigmoid (real / fake)
    """
    def __init__(self, vocab_size, embed_dim=64, num_filters=128,
                 seq_len=MAX_SEQ_LEN):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        # Multiple kernel sizes to capture n-gram features
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k, padding=k // 2)
            for k in [3, 4, 5]
        ])
        self.dropout = nn.Dropout(0.3)
        self.fc      = nn.Linear(num_filters * 3, 1)

    def forward(self, tokens):
        # tokens: (B, T)
        x = self.embed(tokens).permute(0, 2, 1)     # (B, E, T)
        pooled = []
        for conv in self.convs:
            c = torch.relu(conv(x))                  # (B, F, T')
            c = c.max(dim=2).values                  # (B, F)
            pooled.append(c)
        x = torch.cat(pooled, dim=1)                 # (B, F*3)
        x = self.dropout(x)
        return torch.sigmoid(self.fc(x))             # (B, 1)

In [ ]:
NOISE_DIM  = 100
EMBED_DIM  = 64
HIDDEN_DIM = 128
NUM_FILTERS = 128
LR_G = 2e-4
LR_D = 1e-4

G = Generator(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    seq_len=MAX_SEQ_LEN,
    noise_dim=NOISE_DIM
).to(DEVICE)

D = Discriminator(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    num_filters=NUM_FILTERS,
    seq_len=MAX_SEQ_LEN
).to(DEVICE)

opt_G = optim.Adam(G.parameters(), lr=LR_G, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=LR_D, betas=(0.5, 0.999))

criterion = nn.BCELoss()

print(f"Generator params    : {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")

In [ ]:
def real_labels(size):
    # Smooth real labels: 0.9 instead of 1.0 — reduces overconfident D
    return torch.full((size, 1), 0.9, device=DEVICE)

def fake_labels(size):
    return torch.zeros(size, 1, device=DEVICE)

In [ ]:
EPOCHS        = 30
D_STEPS       = 2       # train D this many steps before 1 G step
LOG_INTERVAL  = 5

history = {
    "epoch": [],
    "D_loss": [], "G_loss": [],
    "D_real_acc": [], "D_fake_acc": []
}

for epoch in range(1, EPOCHS + 1):
    G.train(); D.train()
    epoch_D_loss = 0.0
    epoch_G_loss = 0.0
    epoch_D_real_acc = 0.0
    epoch_D_fake_acc = 0.0
    n_batches = 0

    for real_batch in train_loader:
        real_batch = real_batch.to(DEVICE)
        bsz = real_batch.size(0)

        # ── Train Discriminator ──────────────────────────────────────
        for _ in range(D_STEPS):
            opt_D.zero_grad()

            # Real
            real_pred = D(real_batch)
            loss_real = criterion(real_pred, real_labels(bsz))

            # Fake
            noise = torch.randn(bsz, NOISE_DIM, device=DEVICE)
            with torch.no_grad():
                _, fake_tokens = G(noise)
            fake_pred = D(fake_tokens.detach())
            loss_fake = criterion(fake_pred, fake_labels(bsz))

            loss_D = (loss_real + loss_fake) / 2
            loss_D.backward()
            opt_D.step()

        # ── Train Generator ──────────────────────────────────────────
        opt_G.zero_grad()

        noise = torch.randn(bsz, NOISE_DIM, device=DEVICE)
        _, fake_tokens = G(noise)
        fake_pred_for_G = D(fake_tokens)

        # Generator wants D to call fakes real
        loss_G = criterion(fake_pred_for_G, real_labels(bsz))
        loss_G.backward()
        opt_G.step()

        # ── Accumulate metrics ───────────────────────────────────────
        epoch_D_loss     += loss_D.item()
        epoch_G_loss     += loss_G.item()
        epoch_D_real_acc += (real_pred >= 0.5).float().mean().item()
        epoch_D_fake_acc += (fake_pred < 0.5).float().mean().item()
        n_batches += 1

    # Averages
    avg_D = epoch_D_loss     / n_batches
    avg_G = epoch_G_loss     / n_batches
    avg_Dr = epoch_D_real_acc / n_batches
    avg_Df = epoch_D_fake_acc / n_batches

    history["epoch"].append(epoch)
    history["D_loss"].append(avg_D)
    history["G_loss"].append(avg_G)
    history["D_real_acc"].append(avg_Dr)
    history["D_fake_acc"].append(avg_Df)

    if epoch % LOG_INTERVAL == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}/{EPOCHS} | "
              f"D_loss: {avg_D:.4f} | G_loss: {avg_G:.4f} | "
              f"D(real): {avg_Dr:.3f} | D(fake): {avg_Df:.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history["epoch"], history["D_loss"], label="D loss", color="#e74c3c")
ax1.plot(history["epoch"], history["G_loss"], label="G loss", color="#3498db")
ax1.set_title("Generator vs Discriminator Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("BCE Loss")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history["epoch"], history["D_real_acc"],
         label="D acc on real", color="#2ecc71")
ax2.plot(history["epoch"], history["D_fake_acc"],
         label="D acc on fake", color="#e67e22")
ax2.axhline(0.5, linestyle="--", color="grey", alpha=0.6, label="0.5 baseline")
ax2.set_title("Discriminator Accuracy")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("results/training_histories/gan_training_history.png", dpi=150)
plt.show()

pd.DataFrame(history).to_csv(
    "results/training_histories/gan_training_history.csv", index=False)
print("Saved training history.")

In [ ]:
def generate_samples(n=20, temperature=0.8):
    G.eval()
    samples = []
    with torch.no_grad():
        noise = torch.randn(n, NOISE_DIM, device=DEVICE)
        _, tokens = G(noise, temperature=temperature)
        for seq in tokens.cpu().numpy():
            samples.append(decode(seq))
    return samples

generated = generate_samples(n=20, temperature=0.8)

print("=== Generated Financial Tweets ===")
for i, s in enumerate(generated, 1):
    print(f"{i:>2}. {s}")

In [ ]:
os.makedirs("results/sample_outputs", exist_ok=True)

with open("results/sample_outputs/gan_generated_samples.txt", "w") as f:
    f.write("=== GAN Generated Financial Tweets ===\n\n")
    for i, s in enumerate(generated, 1):
        f.write(f"{i}. {s}\n")

print("Saved: results/sample_outputs/gan_generated_samples.txt")

In [ ]:
G.eval(); D.eval()

all_preds  = []
all_labels = []

# Real tweets from test set
test_dataset = TweetDataset(test_encoded)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        preds = D(batch)
        all_preds.extend((preds >= 0.5).long().cpu().numpy().flatten())
        all_labels.extend([1] * len(batch))   # 1 = real

    # Fake tweets
    noise = torch.randn(len(test_encoded), NOISE_DIM, device=DEVICE)
    _, fake_tokens = G(noise)
    fake_preds = D(fake_tokens)
    all_preds.extend((fake_preds >= 0.5).long().cpu().numpy().flatten())
    all_labels.extend([0] * len(test_encoded))  # 0 = fake

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print("=== Discriminator Classification Report ===")
print(classification_report(all_labels, all_preds,
                             target_names=["Fake", "Real"]))

prec, rec, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average="macro")
print(f"Macro Precision: {prec:.4f}")
print(f"Macro Recall   : {rec:.4f}")
print(f"Macro F1       : {f1:.4f}")

In [ ]:
smoother = SmoothingFunction().method1

# References: list of list of tokens
references = [word_tokenize(t.lower()) for t in test_df["clean_text"]]
references_wrapped = [[r] for r in references]  # corpus_bleu format

# Hypotheses: generated sentences
hypotheses = [word_tokenize(s.lower()) for s in generated]

# Pad hypotheses list to same length as references for corpus BLEU
# (we sample a subset of references equal to len(hypotheses))
ref_subset = references_wrapped[:len(hypotheses)]

bleu = corpus_bleu(ref_subset, hypotheses,
                   smoothing_function=smoother)
print(f"Generator BLEU Score: {bleu:.4f}")

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

# Compare each generated sentence against a random real reference
np.random.seed(42)
ref_sample = test_df["clean_text"].sample(
    n=len(generated), random_state=42).values

for gen, ref in zip(generated, ref_sample):
    scores = scorer.score(ref, gen)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

print(f"Generator ROUGE-1 : {np.mean(rouge1_scores):.4f}")
print(f"Generator ROUGE-2 : {np.mean(rouge2_scores):.4f}")
print(f"Generator ROUGE-L : {np.mean(rougeL_scores):.4f}")

In [ ]:
print("\n" + "=" * 50)
print("TEXT GAN — PERFORMANCE SUMMARY")
print("=" * 50)
print(f"\n--- Discriminator (Classification) ---")
print(f"  Macro Precision : {prec:.4f}")
print(f"  Macro Recall    : {rec:.4f}")
print(f"  Macro F1        : {f1:.4f}")
print(f"\n--- Generator (Generation Quality) ---")
print(f"  BLEU            : {bleu:.4f}")
print(f"  ROUGE-1         : {np.mean(rouge1_scores):.4f}")
print(f"  ROUGE-2         : {np.mean(rouge2_scores):.4f}")
print(f"  ROUGE-L         : {np.mean(rougeL_scores):.4f}")
print(f"\n--- Training ---")
print(f"  Epochs          : {EPOCHS}")
print(f"  Final D loss    : {history['D_loss'][-1]:.4f}")
print(f"  Final G loss    : {history['G_loss'][-1]:.4f}")
print("=" * 50)

# Save to JSON for the comparison matrix
gan_metrics = {
    "discriminator": {"precision": prec, "recall": rec, "f1": f1},
    "generator": {
        "bleu": bleu,
        "rouge1": float(np.mean(rouge1_scores)),
        "rouge2": float(np.mean(rouge2_scores)),
        "rougeL": float(np.mean(rougeL_scores))
    },
    "training": {
        "epochs": EPOCHS,
        "final_D_loss": history["D_loss"][-1],
        "final_G_loss": history["G_loss"][-1]
    }
}

with open("results/gan_metrics.json", "w") as f:
    json.dump(gan_metrics, f, indent=2)

print("\nSaved: results/gan_metrics.json")